### rag pipeline - data ingestion to vectorDB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

d:\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
###read all the pdfs inisde the directory
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #find all pdfs recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} pdf files to process")
    for pdf_file in pdf_files:
        print(f"\nProcessing : {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file)) 
            documents=loader.load()

            ##add source info to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f" error : {e}")
        
    print(f"\n total doc loaded:{len(all_documents)}")
    return all_documents
all_pdf_documents=process_all_pdfs("../data")

Found 3 pdf files to process

Processing : Indrasena_2548561_ETE-DL QP.pdf
loaded 16 pages

Processing : Indrasena_561_NLP_ETE.pdf
loaded 11 pages

Processing : Satwik resume.pdf
loaded 2 pages

 total doc loaded:29


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Indrasena_2548561_ETE-DL QP', 'source': '..\\data\\pdfs\\Indrasena_2548561_ETE-DL QP.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Indrasena_2548561_ETE-DL QP.pdf', 'file_type': 'pdf'}, page_content='CHRIST  (Deemed  to  be  University),  Bangalore  –  560  029   Department  of  Computer  Science   END-TRIMESTER  EXAMINATION  MARCH-  2026   PG  III  Trimester   \nProgramme  Name:  MSAIM  Max.  Marks:  100  Course  Name:  -  Deep  Learning  Time:  3  Hrs  \nCourse\n \nCode:\n \nMAI417-3\n \n \nGeneral  Instructions   ●  All  rough  work  should  be  done  in  the  answer  script.  Do  not  write  or  scribble  on  the  question  paper  \nexcept\n \nyour\n \nregistration\n \nnumber.\n \n ●  Verify  the  Course  code  /  Course  title  &  number  of  pages  of  questions  in  the  question  paper.  \n●\n \nEnsure\n \nyour\n \nmobile\n \nphone\n \

In [8]:
##textsplitting into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split{len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"example chunk")
        print(f"content:{split_docs[0].page_content[:200]}...")
        print(f"metdata:{split_docs[0].metadata}")
    return split_docs

In [9]:
chunks = split_documents(all_pdf_documents)

Split29 documents into 38 chunks
example chunk
content:CHRIST  (Deemed  to  be  University),  Bangalore  –  560  029   Department  of  Computer  Science   END-TRIMESTER  EXAMINATION  MARCH-  2026   PG  III  Trimester   
Programme  Name:  MSAIM  Max.  Mark...
metdata:{'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Indrasena_2548561_ETE-DL QP', 'source': '..\\data\\pdfs\\Indrasena_2548561_ETE-DL QP.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Indrasena_2548561_ETE-DL QP.pdf', 'file_type': 'pdf'}


## embedding and vectorstore db

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Any,Tuple,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
## handles the embedding generation using setence transformer
class EmbeddingManager:
    def __init__(self,model_name:str = "all-MiniLM-L6-v2"):
        """
        intilaize the embedding manager
        Args:
            Model name: hugging face model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()## its going to load the model
     ## loading the sentence transformer   
    def _load_model(self):# why we are using _ in the begginig.. it is a protected functionand it will be accessible only inside this class
        try:
            print(f"loading embedding model{self.model_name}")
            self.model =SentenceTransformer(self.model_name)
            print("model loaded successfully. embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}:{e}")
            raise

    ## generating the embedding for the list of texts
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """
        ARGS:
            texts:list of text strings to emb
        return:
            numpy array od embeddings wiht shape (len(texts),embedding_dim)
        
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"generating embeddings for {len(texts)}texts")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"generated embeddings wiht shape :{embeddings.shape}")
        return embeddings

##def get_embedding_dimension(self) ->int:
# ##   """get the embedding dimesnion of the model"""
# ## if not self.model():
# ## return self.model.get_sentence_embedding_dimension()
# ##     raise valueError("model not loaded")


## intilaize the embedding manager
embedding_manager=EmbeddingManager()

loading embedding modelall-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3732.33it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully. embedding dimension: {self.model.get_sentence_embedding_dimension()}
